# SpeechBrain x-vector + K-means 화자 분리

사전학습 x-vector 임베딩을 추출하고 K-means로 군집하여 화자 분리(diarization)를 수행합니다.

## 의존성 설치

In [ ]:
# pip install numpy torchaudio speechbrain

## 라이브러리 import

In [ ]:
import numpy as np
import torch
import torchaudio
from speechbrain.pretrained import SpeakerRecognition
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

## XVectorDiarizer 클래스 정의

In [ ]:
class XVectorDiarizer:
    def __init__(self, model_source="speechbrain/spkrec-xvect-voxceleb",
                 seg_len=1.5, hop=0.75, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.verification = SpeakerRecognition.from_hparams(
            source=model_source,
            savedir="pretrained_models/spkrec-xvect-voxceleb",
            run_opts={"device": self.device},
        )
        self.seg_len = seg_len
        self.hop = hop

    def _load(self, audio_path, sr=16000):
        wav, file_sr = torchaudio.load(audio_path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if file_sr != sr:
            wav = torchaudio.functional.resample(wav, file_sr, sr)
        return wav.squeeze(0), sr

    def _segment(self, wav, sr):
        seg_samples = int(self.seg_len * sr)
        hop_samples = int(self.hop * sr)
        segs, bounds = [], []
        for start in range(0, len(wav) - seg_samples + 1, hop_samples):
            segs.append(wav[start:start + seg_samples])
            bounds.append((start / sr, (start + seg_samples) / sr))
        return segs, bounds

    def extract_embeddings(self, audio_path):
        wav, sr = self._load(audio_path)
        segs, bounds = self._segment(wav, sr)
        if not segs:
            raise ValueError("오디오가 세그먼트 길이보다 짧습니다.")
        emb_list = []
        with torch.no_grad():
            for seg in segs:
                seg = seg.unsqueeze(0).to(self.device)
                emb = self.verification.encode_batch(seg)
                emb_list.append(emb.squeeze().cpu().numpy())
        return np.array(emb_list), bounds

    def diarize(self, audio_path, n_speakers=2, random_state=42):
        embeddings, bounds = self.extract_embeddings(audio_path)
        scaler = StandardScaler()
        X = scaler.fit_transform(embeddings)
        kmeans = KMeans(n_clusters=n_speakers, random_state=random_state, n_init=20)
        labels = kmeans.fit_predict(X)
        return labels, bounds


def format_timeline(labels, bounds):
    lines, prev = [], None
    for label, (t0, t1) in zip(labels, bounds):
        line = f"[{t0:6.2f}s - {t1:6.2f}s] 화자 {label}"
        if label != prev:
            lines.append(line)
            prev = label
        else:
            lines[-1] = line
    return "\n".join(lines)

## 데모: 합성 오디오로 2명 화자 시뮬레이션

외부 파일 없이 노트북만으로 동작을 확인하기 위해 서로 다른 주파수의 톤을 섞은 합성 오디오를 생성합니다. (실제 화자 구별은 아니며 파이프라인 검증용)

In [ ]:
def make_demo_audio(path="demo.wav", sr=16000, duration=12.0):
    t = torch.linspace(0, duration, int(sr * duration))
    # 0~6초: 220Hz 톤(화자 A), 6~12초: 330Hz 톤(화자 B)
    sig = torch.where(
        t < duration / 2,
        torch.sin(2 * np.pi * 220 * t),
        torch.sin(2 * np.pi * 330 * t),
    )
    sig = (sig * 0.3).unsqueeze(0)
    torchaudio.save(path, sig, sr)
    return path

demo_path = make_demo_audio()
print("데모 오디오 생성:", demo_path)

## 화자 분리 실행

In [ ]:
diarizer = XVectorDiarizer()
# 실제 파일 사용 시: labels, bounds = diarizer.diarize("audio.wav", n_speakers=2)
labels, bounds = diarizer.diarize(demo_path, n_speakers=2)
print(format_timeline(labels, bounds))

## 임베딩 시각화 (PCA 2차원)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

embs, _ = diarizer.extract_embeddings(demo_path)
X = StandardScaler().fit_transform(embs)
xy = PCA(n_components=2).fit_transform(X)

plt.figure(figsize=(6, 5))
plt.scatter(xy[:, 0], xy[:, 1], c=labels, cmap="tab10")
plt.title("x-vector 임베딩 PCA (색=K-means 화자)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()